# LIBERO — **옛 50ep 정리 + 로그에서 500ep eval_info 복구**

옛날 **50ep** eval 잔재(action·eval_info)를 지워 깔끔한 **500ep** 만 남기고,
eval 은 끝났는데 `eval_info.json` 이 안 남은 것(seed1·3)을 **로그에서 SR 복구**한다.

판별(실제 로그로 확인): LIBERO-10 = 10 task → **정상 500ep = overall n_ep 5000**, 옛 50ep = 500.
action_logs 파일 수도 새 ~500 / 옛 ~50 (10 task 가 같은 폴더에 덮어써 마지막 task 것만 남음).

- ⚠️ seed1 의 현재 eval_info 는 **옛 50ep 것** → 지우고 새로 복구.
- 순서: **인벤토리 → ① 옛것 제거 → ② 로그 복구**. 둘 다 dry-run→EXECUTE.


In [ ]:
import sys, json, re, shutil
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)

TASK   = 'libero_10'
MODELS = ['act', 'acm', 'acm2', 'mosaic', 'bimamba', 'bimamba_s7']
SEEDS  = [0, 1, 2, 3]
N_TASKS  = 10                     # LIBERO-10 = 10 task
PER_TASK = 500                    # 정상 = task당 500ep
NEW_TOTAL = N_TASKS * PER_TASK    # overall n_ep = 5000 (정상 500ep). 옛 50ep = 500.
MIN_NEW_EP  = NEW_TOTAL // 2      # overall n_ep 이 이 값 이상 = 새(500ep). 미만 = 옛(50ep).
MIN_NEW_ACT = PER_TASK // 2       # action_logs 파일 수가 이 값 이상 = 새. 미만 = 옛.

EVAL_ROOT = cf.OUTPUT_BASE / 'eval_clean' / TASK
LOG_DIR   = cf.OUTPUT_BASE / '_logs'
print('eval:', EVAL_ROOT, '| logs:', LOG_DIR)
print(f'판별: 새(500ep)=overall n_ep>={MIN_NEW_EP} · action>={MIN_NEW_ACT} / 옛(50ep)=그 미만')

## 인벤토리 — seed 마다 action(개수)·eval_info(n_ep) 를 새/옛 분류


In [ ]:
# ── 인벤토리: seed 마다 action_logs 폴더(파일 수)·eval_info(n_ep) 를 새/옛 으로 분류 ──
def n_ep_of(info_path):
    try:
        return int(json.loads(info_path.read_text()).get('overall', {}).get('n_episodes') or 0)
    except Exception:
        return 0

plan_remove, need_recover, need_fresh = [], [], []   # (제거대상 경로), (tag,s,new_eval_dir), (tag,s)
print(f"{'model':<12}{'seed':>5}   action_logs(개수)             eval_info(n_ep)")
print('-' * 72)
for tag in MODELS:
    for s in SEEDS:
        d = EVAL_ROOT / tag / f'seed{s}'
        if not d.is_dir():
            print(f'{tag:<12}{s:>5}   (폴더 없음)'); continue
        acts = sorted(d.rglob('action_logs'))
        infos = sorted(d.rglob('eval_info.json'))
        new_act_dirs, old_act_dirs = [], []
        for al in acts:
            n = len(list(al.glob('episode_*.pt')))
            (new_act_dirs if n >= MIN_NEW_ACT else old_act_dirs).append((al, n))
        new_infos, old_infos = [], []
        for inf in infos:
            ne = n_ep_of(inf)
            (new_infos if ne >= MIN_NEW_EP else old_infos).append((inf, ne))
        a_str = ', '.join(f'{al.parent.parent.name}:{n}{"(옛)" if n < MIN_NEW_ACT else ""}' for al, n in new_act_dirs + old_act_dirs) or '-'
        i_str = ', '.join(f'{ne}{"(옛)" if ne < MIN_NEW_EP else ""}' for _, ne in new_infos + old_infos) or '없음'
        print(f'{tag:<12}{s:>5}   {a_str:<28} {i_str}')
        # 옛것 제거 목록
        for al, n in old_act_dirs:
            plan_remove.append(al)                 # 옛 action_logs 폴더
        for inf, ne in old_infos:
            plan_remove.append(inf)                # 옛 eval_info.json
        # 새 action 있는데 새 eval_info 없으면 → 로그 복구 대상
        if new_act_dirs and not new_infos:
            new_eval_dir = new_act_dirs[0][0].parent.parent   # …/actions/action_logs → …/rep0
            need_recover.append((tag, s, new_eval_dir))
        # 새 action 이 아예 없으면 → 새로 eval 해야 함
        if not new_act_dirs:
            need_fresh.append((tag, s))

print('\n' + '=' * 72)
print('■ 제거할 옛 50ep (action_logs·eval_info):', len(plan_remove), '개')
print('■ 로그에서 eval_info 복구 대상:', [f'{t}/s{s}' for t, s, _ in need_recover] or '없음')
print('■ 새로 eval 필요(새 action 없음):', [f'{t}/s{s}' for t, s in need_fresh] or '없음')

## ① 옛 50ep 제거 (dry-run → EXECUTE=True)
action_logs 폴더(~50개)·eval_info(n_ep<2500) 만. **새(500ep) 것은 안 건드림.**


In [ ]:
# ── ① 옛 50ep 데이터 제거 (action_logs 폴더 + eval_info.json) ── 확인 후 EXECUTE=True ──
EXECUTE = False

if not plan_remove:
    print('제거할 옛 50ep 없음.')
else:
    print(f'{"제거" if EXECUTE else "DRY-RUN"} — 옛 50ep {len(plan_remove)}개:')
    for p in plan_remove:
        kind = 'DIR ' if p.is_dir() else 'FILE'
        print(f'   {kind} {p.relative_to(EVAL_ROOT)}')
        if EXECUTE:
            shutil.rmtree(p) if p.is_dir() else p.unlink()
    print('\n' + ('제거 완료.' if EXECUTE else '확인됐으면 EXECUTE=True 로 다시 실행.'))

## ② 로그에서 500ep eval_info 복구 (dry-run → EXECUTE=True)
새 action 있는데 새 eval_info 없는 seed → 로그의 overall(n_ep≈5000) SR 로 eval_info.json 생성.


In [ ]:
# ── ② 로그에서 새(500ep) SR 복구 → eval_info.json 생성 ── 확인 후 EXECUTE=True ──
EXECUTE = False

def logs_for(tag, seed):
    key = f'{tag}__seed{seed}'
    return sorted(p for p in LOG_DIR.glob('*.log') if key in p.name) if LOG_DIR.is_dir() else []

def new_result_from_logs(tag, seed):
    # 로그에서 (pc_success, n_episodes) 쌍을 모두 뽑아, 새(500ep=overall≈5000) 중 마지막 것.
    best = None
    for lp in logs_for(tag, seed):
        txt = lp.read_text(errors='ignore')
        for m in re.finditer(r"pc_success['\"]?\s*[:=]\s*([0-9]+\.?[0-9]*),\s*['\"]?n_episodes['\"]?\s*[:=]\s*([0-9]+)", txt):
            sr, ne = float(m.group(1)), int(m.group(2))
            if ne >= MIN_NEW_EP:                       # 새(500ep) 결과만
                best = (sr, ne, lp)                    # 마지막(가장 최근) 것으로 갱신
    return best

if not need_recover:
    print('복구 대상 없음.')
else:
    print(f'{"복원" if EXECUTE else "DRY-RUN"} — 로그→eval_info:')
    for tag, s, new_eval_dir in need_recover:
        res = new_result_from_logs(tag, s)
        if res is None:
            print(f'   {tag}/seed{s}: 로그에서 500ep 결과 못 찾음 → 재eval 필요')
            continue
        sr, ne, lp = res
        target = new_eval_dir / 'eval_info.json'
        print(f'   {tag}/seed{s}: SR={sr:.2f}% n_ep={ne} (per-task {ne//N_TASKS}) → {target.relative_to(EVAL_ROOT)}  [{lp.name}]')
        if EXECUTE and not target.exists():
            target.write_text(json.dumps(
                {'overall': {'pc_success': sr, 'n_episodes': ne}, '_recovered_from_log': lp.name}, indent=2))
    print('\n' + ('복원 완료 → eval_final 재실행하면 반영.' if EXECUTE else '확인됐으면 EXECUTE=True.'))